This file was created partially with GeminiAI - I created a first draft using gemini, then modified it manually. 

Unfortunately, I neglected to keep the full chat transcript, so I am unable to include it here (I'm still getting used to a heavily GenAI approach to coding, but as a initial practice toward repeatability in research, the first and most fundamental step is to keep and include the transcript as a coding artifact Because this workbook is a technical illustration not related to a research study or real world project, and all data is fabricated, I don't think the impact in this case is especially serious, but I want to get and stay in the habit, so I do apologize for not including it here...

As a general summary, I asked gemini to create a python-based jupyter notebook to create a topic model using gensim for a collection of medical research related texts with three distinct topics. I asked gensim to simulate the text for these documents (so this is not, to my knowledge, based on actual research), then create a visual output where we can view the clusters. I did a few iterations to try to get a little overlap between the clusters (gensim responded, properly I think, by trying different arrangements of sample text day to generate more overlap in the topics). Lastly, I asked for a visualization using gensim to interactively explore the resulting topic clusters. 

I was only partially successful with this, and because this is an illustration, I eventually moved on. But one good way to understand code is to change it, adapt it, re-run it, and try to make it do something different. So it might be a good exercise for you to see if you can get more overlap. Alternatively you could try running this data through a different clustering (non-supervised ML) algorithm. This approach to verifying code will become more important as an increasing amount of code is initially written, at least as a first draft, by GenAI based systems. 

In [1]:
import gensim
import gensim.corpora as corpora
from gensim.models import LdaModel
from gensim.models import CoherenceModel
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
import warnings


# Suppress warnings for cleaner output, especially from pyLDAvis
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 1. Download NLTK data (if not already downloaded) ---

nltk.download('stopwords')

nltk.download('wordnet')

nltk.download('punkt')

# --- 2. Prepare Sample Text Data ---
# In a real scenario, this would be loaded from files (e.g., articles, reviews)
documents = [
"""Cardiovex improves vascular tone and reduces systemic inflammation
through calcium signaling modulation.""",
"""A nanoparticle formulation of Cardiovex enhances delivery to cardiac
and neural tissues, reducing off-target effects.""",
"""Cardiovex combined with immune modulators reduces neuroinflammatory
markers in aged animal models.""",
"""Neurodaptin targets calcium ion channels implicated in both synaptic
transmission and cardiac excitability.""",
"""A sustained-release formulation of Neurodaptin improves cognitive
outcomes and reduces vascular inflammation.""",
"""Co-administration of Neurodaptin and Immunexor shows promise in
reducing neuroinflammation and enhancing immune response.""",
"""Immunexor regulates T-cell activation and reduces neuroinflammation
in models of chronic cognitive impairment.""",
"""Encapsulated Immunexor nanoparticles target vascular endothelium and
lymphoid tissues for dual immune-cardiovascular effects.""",
"""Combined use of Immunexor and Cardiovex enhances tumor suppression by
modulating both inflammatory and vascular pathways."""
]




# --- 3. Text Preprocessing ---
# Tokenization, Lowercasing, Stopword Removal, Lemmatization

# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Tokenizes, lowercases, removes stopwords, and lemmatizes text.
    """
    tokens = word_tokenize(text.lower())
    # Remove non-alphabetic tokens and stopwords, then lemmatize
    processed_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token.isalpha() and token not in stop_words
    ]
    return processed_tokens

# Apply preprocessing to all documents
processed_documents = [preprocess_text(doc) for doc in documents]

print("--- Processed Documents (first 2) ---")
for i, doc in enumerate(processed_documents[:2]):
    print(f"Doc {i+1}: {doc}")
print("\n")

# --- 4. Create Gensim Dictionary and Corpus ---
# A dictionary maps each word to a unique ID
dictionary = corpora.Dictionary(processed_documents)

# Filter out words that appear in less than no_below documents
# or in more than no_above fraction of documents (and keep only max_features)
dictionary.filter_extremes(no_below=2, no_above=0.5) # Adjust these parameters based on your corpus

# Convert list of documents (tokens) into a Bag-of-Words (BoW) corpus
# Each document is represented as a list of (word_id, word_count) tuples
corpus = [dictionary.doc2bow(doc) for doc in processed_documents]

print("--- Sample Corpus Entry (first document) ---")
# Example: print the first document in BoW format and its corresponding words
print(f"BoW format: {corpus[0]}")
print("Words in first document:")
for word_id, freq in corpus[0]:
    print(f"  {dictionary[word_id]}: {freq}")
print("\n")

# --- 5. Train the LDA Model ---
# num_topics: The number of topics to generate. This is a crucial parameter.
# id2word: Mapping from word IDs to words (from the dictionary).
# passes: Number of passes through the corpus during training.
# random_state: For reproducibility.
# alpha & eta: Hyperparameters for LDA (often 'auto' or symmetric/asymmetric).
num_topics = 3 # You can experiment with different numbers of topics
lda_model = LdaModel(corpus=corpus,
                     id2word=dictionary,
                     num_topics=num_topics,
                     random_state=100,
                     update_every=1,
                     chunksize=100,
                     passes=10,
                     alpha='auto',
                     per_word_topics=True)

print(f"--- LDA Model Training Complete ({num_topics} topics) ---\n")

# --- 6. Display Topics (Top Words per Topic) ---
print("--- Top Words for Each Topic ---")
for idx, topic in lda_model.print_topics(num_words=5):
    print(f"Topic {idx}: {topic}")
print("\n")

# --- 7. Evaluate Model (Optional: Coherence Score) ---
# Coherence score helps evaluate the interpretability of topics
coherence_model_lda = CoherenceModel(model=lda_model,
                                     texts=processed_documents,
                                     dictionary=dictionary,
                                     coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print(f"Coherence Score: {coherence_lda:.4f}\n")

# --- 8. Visualize Topics with pyLDAvis ---
# This generates an interactive HTML visualization
print("--- Generating pyLDAvis Visualization ---")
# Prepare the data for pyLDAvis
# Ensure you use gensimvis.prepare for Gensim models
vis_data = gensimvis.prepare(lda_model, corpus, dictionary)

# Save the visualization to an HTML file
# You can open this file in your web browser to interact with the visualization
pyLDAvis.save_html(vis_data, 'lda_visualization.html')

print("pyLDAvis visualization saved to 'lda_visualization.html'")
print("Open 'lda_visualization.html' in your web browser to view the interactive topic clusters.")


[nltk_data] Downloading package stopwords to /Users/nms-
[nltk_data]     workshop/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/nms-
[nltk_data]     workshop/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /Users/nms-
[nltk_data]     workshop/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


--- Processed Documents (first 2) ---
Doc 1: ['cardiovex', 'improves', 'vascular', 'tone', 'reduces', 'systemic', 'inflammation', 'calcium', 'signaling', 'modulation']
Doc 2: ['nanoparticle', 'formulation', 'cardiovex', 'enhances', 'delivery', 'cardiac', 'neural', 'tissue', 'reducing', 'effect']


--- Sample Corpus Entry (first document) ---
BoW format: [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1)]
Words in first document:
  calcium: 1
  cardiovex: 1
  improves: 1
  inflammation: 1
  reduces: 1
  vascular: 1


--- LDA Model Training Complete (3 topics) ---

--- Top Words for Each Topic ---
Topic 0: 0.097*"reduces" + 0.096*"vascular" + 0.096*"cardiovex" + 0.068*"combined" + 0.068*"neurodaptin"
Topic 1: 0.114*"neuroinflammation" + 0.114*"model" + 0.114*"cognitive" + 0.114*"immunexor" + 0.114*"reduces"
Topic 2: 0.100*"cardiac" + 0.100*"tissue" + 0.100*"effect" + 0.100*"target" + 0.073*"calcium"


Coherence Score: 0.3844

--- Generating pyLDAvis Visualization ---
pyLDAvis visualization 

In [2]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /Users/nms-
[nltk_data]     workshop/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True